 5년간에걸쳐약150개국가에서조사한행복지수와행복에관련이있을것으로예
상되는GDP나건강과같은지수를담고있다. 2015년부터2019년까지조사결과가
각각`2015.csv` ~ `2019.csv` 파일에 들어있다. 매해 조사 국가가 조금씩 차이가 있으
며컬럼가지수나이름또한조금씩다르다. 따라서데이터를분석하기위해컬럼이
름을직접출력해보고확인하여야한다

##Question
-  5년동안 행복지수가 지속적으로 상승하거나 적어도 전년도와 같은 나라를 찾아 출력 하시오

In [1]:
from pyspark import SparkContext
import urllib.request
import urllib.parse

In [2]:
for year in range(2015, 2020):
    path = f"data/World Hapiness Report/{year}.csv"
    encoded_path = urllib.parse.quote(path)

    url = f"https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/{encoded_path}"

    print(url)
    urllib.request.urlretrieve(url, f"{year}.csv")

https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World%20Hapiness%20Report/2015.csv
https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World%20Hapiness%20Report/2016.csv
https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World%20Hapiness%20Report/2017.csv
https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World%20Hapiness%20Report/2018.csv
https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World%20Hapiness%20Report/2019.csv


In [3]:
sc = SparkContext.getOrCreate()

# only need (country, happiness_score)
for year in range(2015, 2020):
    rdd = sc.textFile(f"{year}.csv")
    print(year, rdd.first())

2015 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
2016 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
2017 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Generosity,Trust (Government Corruption)
2018 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
2019 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)


In [4]:
# extract (country, happiness_score)
import csv

scores = {}

for year in range(2015, 2020):
    scores[year] = (
        sc.textFile(f"{year}.csv")
          .map(lambda line: next(csv.reader([line])))  # handles "Hong Kong S.A.R., China"
          .filter(lambda row: row[0] != "Country")  # drop header
          .map(lambda row: (row[0], round(float(row[2]), 3)))  # round: 2.904999971 -> 2.905
    )

In [5]:
#join countries scores across years

joined = scores[2015].join(scores[2016]).join(scores[2017]).join(scores[2018]).join(scores[2019])

In [6]:
#clean up joined

flattened = joined.map(
    lambda x: (
        x[0],
        (
            x[1][0][0][0][0],
            x[1][0][0][0][1],
            x[1][0][0][1],
            x[1][0][1],
            x[1][1]
        )
    )
)

In [7]:
#행복지수가 지속적으로 상승하거나 적어도 전년도와 같은 나라

result = flattened.filter(
    lambda x:
        x[1][0] <= x[1][1]
        <= x[1][2]
        <= x[1][3]
        <= x[1][4]
)

In [8]:
result.map(lambda x: x[0]).collect()

['Gabon',
 'Malta',
 'Senegal',
 'Chad',
 'Czech Republic',
 'Cambodia',
 'Tajikistan',
 'Congo (Brazzaville)',
 'Latvia',
 'Finland',
 'Dominican Republic',
 'Honduras',
 'Syria',
 'Estonia',
 'Mongolia',
 'Poland',
 'Togo',
 'Slovakia',
 'Hungary',
 'Cameroon',
 'Romania',
 'Benin',
 'Burkina Faso',
 'Serbia',
 'Ivory Coast',
 'Philippines',
 'Portugal',
 'Niger',
 'Burundi']

**Note / 참고**

- **Why Burundi is included:** the 2017 file stores scores as 32-bit floats, so 3-decimal scores get tiny errors (2.905 → `2.904999971`, 4.465 → `4.465000153`). Burundi's real scores are 2.905, 2.905, 2.905, 2.905, 3.775: never lower than the year before. Without `round(..., 3)`, the check `2.905 <= 2.904999971` is `False` and Burundi is wrongly excluded. All scores have 3 decimals, so rounding removes only the noise.
  **Burundi를 포함한 이유:** 2017년 파일은 점수를 32비트 float로 저장해서 소수점 셋째 자리 점수에 미세한 오차가 생깁니다 (2.905 → `2.904999971`, 4.465 → `4.465000153`). Burundi의 실제 점수는 2.905, 2.905, 2.905, 2.905, 3.775로 전년도보다 낮아진 적이 없습니다. `round(..., 3)` 없이 비교하면 `2.905 <= 2.904999971`이 `False`가 되어 Burundi가 잘못 제외됩니다. 모든 점수는 소수점 셋째 자리까지이므로 반올림은 오차만 제거합니다.
- Some countries are named differently across years, so `join` treats them as different countries. Taiwan (`Taiwan Province of China`), Trinidad and Tobago (`Trinidad & Tobago`), and Macedonia (`North Macedonia`) would also qualify if their names were matched.
  일부 국가는 연도별로 이름이 달라 `join`에서 다른 나라로 처리됩니다. 이름을 통일하면 Taiwan, Trinidad and Tobago, Macedonia도 조건을 만족합니다.